# TC3-41 — Engenharia de Features: Seleção e Justificativa

**Responsável:** Benito Marculano Ribeiro 

**Sprint:** Fase 2
 
**Jira:** TC3-41

**Objetivo:** selecionar candidatas a features a partir das variáveis disponíveis na `ml_aluno`, registrando fonte, critério técnico, decisão e próxima etapa.

Este notebook é uma entrega de análise e documentação. Ele não implementa pipeline definitivo de pré-processamento, treinamento de modelo, integração física de bases externas ou análise completa de Data Leakage.


## 1. Insumos utilizados

A seleção de features considera os trabalhos anteriores e complementares do projeto:

- **TC3-38 — Auditoria da Feature Table `ml_aluno`**: valida estrutura, volume, tipos, nulos, granularidade, target e split.
- **TC3-40 — Catálogo de Variáveis e Fontes de Dados Adicionais**: documenta as variáveis da `ml_aluno` e identifica fontes complementares candidatas.
- **TC3-42 — Análise de Risco de Data Leakage**: etapa posterior para investigar variáveis que possam revelar direta ou indiretamente o target.

A Feature Table `ml_aluno` permanece como a base principal da modelagem supervisionada. Censo Escolar 2023 e INSE 2023 são tratados aqui como fontes candidatas futuras, pois o TC3-40 documenta as bases e chaves, mas não realiza a integração física neste notebook.


## 2. Evidências confirmadas sobre a `ml_aluno`

A auditoria TC3-38 registrou os seguintes pontos principais:

- Base consolidada com **3.867.999 registros** e **19 variáveis**.
- Partição 2023 com **1.747.439 registros** e partição 2024 com **2.120.560 registros**.
- Granularidade validada como **1 linha = 1 aluno por ano de referência**.
- `id_aluno` não é chave única global: há **2.352.328 alunos únicos**, sendo **1.515.671** presentes em dois anos.
- Target `alfabetizado` balanceado: classe 0 com **48,69%** e classe 1 com **51,31%**.
- `dataset_split`: treino **69,94%**, validação **15,02%**, teste **15,04%**.
- Variáveis com nulos relevantes: `media_portugues_municipio_ano_anterior`, `taxa_alfabetizacao_municipio_ano_anterior`, `peso_aluno` e `proficiencia`.

Essas evidências não definem sozinhas a seleção final, mas orientam as decisões registradas neste notebook.


## 3. Leitura do catálogo TC3-40

Antes de definir qualquer decisão de feature selection, este notebook lê o catálogo produzido na etapa TC3-40.

O arquivo `docs/data_dictionary/ml_aluno_catalogo_variaveis.csv` contém a lista de variáveis da `ml_aluno`, seus tipos, descrições e categorias. Ele é a fonte de entrada para a análise do TC3-41.

A partir dele, este notebook acrescenta apenas a avaliação analítica: `decisao`, `criterio_principal`, `justificativa` e `proxima_etapa`.


In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 160)


def encontrar_raiz_projeto():
    atual = Path.cwd().resolve()
    for candidato in [atual, *atual.parents]:
        catalogo = candidato / "docs" / "data_dictionary" / "ml_aluno_catalogo_variaveis.csv"
        if catalogo.exists():
            return candidato
    raise FileNotFoundError(
        "Não foi possível localizar docs/data_dictionary/ml_aluno_catalogo_variaveis.csv. "
        "Execute o notebook dentro do repositório ou gere o catálogo no TC3-40."
    )


PROJECT_ROOT = encontrar_raiz_projeto()
catalogo_path = PROJECT_ROOT / "docs" / "data_dictionary" / "ml_aluno_catalogo_variaveis.csv"

catalogo_ml_aluno = pd.read_csv(catalogo_path)
print(f"Catálogo carregado de: {catalogo_path}")
print(f"Linhas: {catalogo_ml_aluno.shape[0]} | Colunas: {catalogo_ml_aluno.shape[1]}")

catalogo_ml_aluno


## 4. Critérios de decisão

As decisões abaixo não são cálculos automáticos. Elas são uma classificação técnica baseada no catálogo, na auditoria TC3-38 e no papel esperado de cada variável dentro de um modelo supervisionado.

Para evitar decisões implícitas, cada variável será ligada a um critério principal.


In [ ]:
criterios_decisao = pd.DataFrame([
    {
        "criterio": "Característica disponível antes da predição",
        "pergunta_orientadora": "A informação existe antes do momento em que o modelo faria a previsão?",
        "decisao_tendencia": "Incluir ou Transformar"
    },
    {
        "criterio": "Identificador ou chave técnica",
        "pergunta_orientadora": "A variável identifica pessoa, escola, município ou processo, sem ser uma característica diretamente generalizável?",
        "decisao_tendencia": "Excluir, Transformar ou usar como chave"
    },
    {
        "criterio": "Variável categórica que exige tratamento",
        "pergunta_orientadora": "A variável precisa de encoding, agregação ou validação de cardinalidade antes da modelagem?",
        "decisao_tendencia": "Transformar"
    },
    {
        "criterio": "Possível proximidade com target ou avaliação",
        "pergunta_orientadora": "A variável pode carregar informação do processo que mede ou define o target?",
        "decisao_tendencia": "Investigar no TC3-42"
    },
    {
        "criterio": "Controle técnico ou variável auxiliar",
        "pergunta_orientadora": "A variável serve para rastreabilidade, separação de treino/teste ou controle interno?",
        "decisao_tendencia": "Excluir da lista de features"
    },
    {
        "criterio": "Redundância operacional",
        "pergunta_orientadora": "A variável duplica informação já representada por outra coluna mais adequada?",
        "decisao_tendencia": "Excluir ou manter apenas para leitura humana"
    }
])

criterios_decisao


## 5. Decisões por variável

A matriz abaixo registra a decisão preliminar do TC3-41 para cada variável da `ml_aluno`.

Categorias de decisão:

- **Incluir:** candidata a entrada do modelo, com uso direto ou tratamento simples posterior.
- **Excluir:** não deve ser usada como feature por representar target, controle técnico, identificador individual, redundância ou informação inadequada.
- **Transformar:** candidata, mas exige encoding, agregação, imputação, normalização ou uso como chave de enriquecimento.
- **Investigar:** tem potencial, mas demanda validação adicional, principalmente quanto a Data Leakage ou disponibilidade no momento de predição.


In [ ]:
decisoes_tc3_41 = pd.DataFrame([
    {
        "Variável": "id_aluno",
        "decisao": "Excluir",
        "criterio_principal": "Identificador ou chave técnica",
        "justificativa": "Identificador individual do aluno. Não deve ser usado diretamente como feature por induzir memorização e não representar característica generalizável.",
        "proxima_etapa": "Usar apenas para auditoria, validação de granularidade e controle de agrupamentos."
    },
    {
        "Variável": "id_escola",
        "decisao": "Transformar",
        "criterio_principal": "Identificador ou chave técnica",
        "justificativa": "Identifica a escola e permite ligação com Censo Escolar e INSE. Como código puro, pode gerar alta cardinalidade e memorização se usado diretamente.",
        "proxima_etapa": "Preferir uso como chave de enriquecimento ou agregações escolares; avaliar encoding adequado se mantido."
    },
    {
        "Variável": "id_municipio",
        "decisao": "Transformar",
        "criterio_principal": "Identificador ou chave técnica",
        "justificativa": "Representa contexto territorial e permite integração municipal. Código categórico de alta cardinalidade exige tratamento antes da modelagem.",
        "proxima_etapa": "Usar como chave para indicadores municipais ou aplicar encoding controlado."
    },
    {
        "Variável": "nome_municipio",
        "decisao": "Excluir",
        "criterio_principal": "Redundância operacional",
        "justificativa": "Nome textual redundante em relação a `id_municipio`, com risco de inconsistência de escrita e sem ganho frente ao código oficial.",
        "proxima_etapa": "Manter apenas para leitura humana e validações."
    },
    {
        "Variável": "sigla_uf",
        "decisao": "Transformar",
        "criterio_principal": "Variável categórica que exige tratamento",
        "justificativa": "Captura diferenças regionais e estaduais relevantes para desempenho educacional. É categórica e precisa de encoding.",
        "proxima_etapa": "Aplicar one-hot encoding ou outra codificação categórica em etapa posterior."
    },
    {
        "Variável": "ano_referencia",
        "decisao": "Incluir",
        "criterio_principal": "Característica disponível antes da predição",
        "justificativa": "Representa o ano da observação e pode capturar efeitos temporais entre 2023 e 2024, respeitando a granularidade validada.",
        "proxima_etapa": "Avaliar uso como variável temporal discreta ou controle de safra."
    },
    {
        "Variável": "serie",
        "decisao": "Incluir",
        "criterio_principal": "Característica disponível antes da predição",
        "justificativa": "Série do aluno é uma característica educacional relevante e disponível antes da avaliação.",
        "proxima_etapa": "Avaliar tratamento como ordinal ou categórica."
    },
    {
        "Variável": "rede",
        "decisao": "Transformar",
        "criterio_principal": "Variável categórica que exige tratamento",
        "justificativa": "Tipo/código da rede de ensino pode refletir diferenças administrativas relevantes, mas deve ser tratado como categoria.",
        "proxima_etapa": "Confirmar domínio de valores e aplicar encoding categórico."
    },
    {
        "Variável": "rede_descricao",
        "decisao": "Excluir",
        "criterio_principal": "Redundância operacional",
        "justificativa": "Descrição textual redundante com `rede`. Usar simultaneamente código e descrição duplicaria informação.",
        "proxima_etapa": "Manter para documentação; usar `rede` como representação modelável."
    },
    {
        "Variável": "caderno",
        "decisao": "Investigar",
        "criterio_principal": "Possível proximidade com target ou avaliação",
        "justificativa": "Pode representar versão/caderno da avaliação. A utilidade depende de ser informação disponível e legítima no momento de predição.",
        "proxima_etapa": "Investigar no TC3-42 se carrega informação ligada ao instrumento de medição do target."
    },
    {
        "Variável": "presenca",
        "decisao": "Investigar",
        "criterio_principal": "Possível proximidade com target ou avaliação",
        "justificativa": "Indica presença na avaliação. Pode estar fortemente ligada à existência de proficiência/target e ao processo de mensuração.",
        "proxima_etapa": "Investigar no TC3-42; se aprovada, tratar como variável categórica/binária."
    },
    {
        "Variável": "preenchimento_caderno",
        "decisao": "Investigar",
        "criterio_principal": "Possível proximidade com target ou avaliação",
        "justificativa": "Pode refletir comportamento durante a avaliação e estar próximo do mecanismo que gera o target.",
        "proxima_etapa": "Investigar no TC3-42 antes de qualquer uso como feature."
    },
    {
        "Variável": "proficiencia",
        "decisao": "Investigar",
        "criterio_principal": "Possível proximidade com target ou avaliação",
        "justificativa": "Mede desempenho do aluno e pode estar muito próxima da definição do target `alfabetizado`. Não deve ser usada sem validação específica de leakage.",
        "proxima_etapa": "Encaminhar ao TC3-42 para confirmar se é insumo permitido ou variável derivada do resultado."
    },
    {
        "Variável": "peso_aluno",
        "decisao": "Investigar",
        "criterio_principal": "Controle técnico ou variável auxiliar",
        "justificativa": "Peso amostral pode ser relevante para avaliação estatística, mas não necessariamente como feature preditiva do aluno.",
        "proxima_etapa": "Avaliar uso como peso/amostra na modelagem ou apenas em análise estatística."
    },
    {
        "Variável": "taxa_alfabetizacao_municipio_ano_anterior",
        "decisao": "Incluir",
        "criterio_principal": "Característica disponível antes da predição",
        "justificativa": "Indicador municipal do ano anterior, com interpretação educacional clara e menor risco temporal que indicadores do próprio ano.",
        "proxima_etapa": "Validar nulos e estratégia de imputação no pré-processamento."
    },
    {
        "Variável": "media_portugues_municipio_ano_anterior",
        "decisao": "Incluir",
        "criterio_principal": "Característica disponível antes da predição",
        "justificativa": "Indicador municipal de desempenho do ano anterior, potencialmente útil para contexto educacional agregado.",
        "proxima_etapa": "Validar nulos e estratégia de imputação no pré-processamento."
    },
    {
        "Variável": "dataset_split",
        "decisao": "Excluir",
        "criterio_principal": "Controle técnico ou variável auxiliar",
        "justificativa": "Indica separação de treino, validação e teste. Usar como feature contaminaria o experimento.",
        "proxima_etapa": "Usar exclusivamente para particionamento e avaliação."
    },
    {
        "Variável": "alfabetizado",
        "decisao": "Excluir",
        "criterio_principal": "Controle técnico ou variável auxiliar",
        "justificativa": "É a variável alvo da modelagem supervisionada, não uma feature de entrada.",
        "proxima_etapa": "Usar exclusivamente como target."
    },
    {
        "Variável": "processed_at",
        "decisao": "Excluir",
        "criterio_principal": "Controle técnico ou variável auxiliar",
        "justificativa": "Timestamp de processamento técnico, sem significado educacional direto para predição.",
        "proxima_etapa": "Manter apenas para rastreabilidade da construção da base."
    }
])

decisoes_tc3_41


## 6. Junção do catálogo com as decisões

A célula abaixo une o catálogo lido do TC3-40 com a matriz de decisões do TC3-41.

A validação `one_to_one` garante que cada variável aparece uma única vez em cada lado da junção. A checagem de nulos garante que nenhuma variável do catálogo ficou sem decisão.


In [ ]:
features_ml_aluno = catalogo_ml_aluno.merge(
    decisoes_tc3_41,
    on="Variável",
    how="left",
    validate="one_to_one"
)

sem_decisao = features_ml_aluno[features_ml_aluno["decisao"].isna()]
if not sem_decisao.empty:
    raise ValueError(
        "Existem variáveis do catálogo sem decisão no TC3-41: "
        + ", ".join(sem_decisao["Variável"].tolist())
    )

features_ml_aluno


## 7. Resumo por decisão


In [ ]:
resumo_decisao = (
    features_ml_aluno
    .groupby("decisao", sort=True)
    .agg(
        quantidade=("Variável", "count"),
        variaveis=("Variável", lambda x: ", ".join(x))
    )
    .reset_index()
)

resumo_decisao


## 8. Features candidatas da base atual

Considerando apenas a `ml_aluno`, as variáveis mais adequadas para compor o conjunto inicial de features são:

- `ano_referencia`
- `serie`
- `sigla_uf`, após encoding
- `rede`, após confirmação do domínio e encoding
- `id_municipio`, preferencialmente como chave para indicadores/agregações ou com encoding controlado
- `id_escola`, preferencialmente como chave para enriquecimento/agregações escolares
- `taxa_alfabetizacao_municipio_ano_anterior`, com imputação posterior
- `media_portugues_municipio_ano_anterior`, com imputação posterior

Variáveis ligadas ao processo de avaliação (`caderno`, `presenca`, `preenchimento_caderno`, `proficiencia`) devem ser analisadas com cuidado no TC3-42 antes de qualquer inclusão definitiva.


## 9. Fontes complementares candidatas

O TC3-40 documenta duas fontes adicionais aprovadas pelo grupo:

| Fonte | Granularidade | Chave escola | Chave município | Uso candidato |
|---|---:|---|---|---|
| Censo Escolar 2023 | Escola | `CO_ENTIDADE` | `CO_MUNICIPIO` | infraestrutura, localização, dependência administrativa e características escolares |
| INSE 2023 | Escola | `ID_ESCOLA` | `CO_MUNICIPIO` | nível socioeconômico e distribuição dos estudantes por níveis INSE |

Essas fontes são candidatas a enriquecimento futuro, mas não estão integradas fisicamente à `ml_aluno` neste notebook. A seleção abaixo é conceitual e deve ser validada em tarefa própria de integração/enriquecimento.


In [ ]:
fontes_complementares = pd.DataFrame([
    {
        "fonte": "Censo Escolar 2023",
        "grupo_de_features": "Infraestrutura escolar, localização, oferta de etapas, dependência administrativa e porte da escola",
        "decisao": "Investigar/Transformar",
        "justificativa": "Pode enriquecer a modelagem com características estruturais da escola, mas possui 408 variáveis e requer seleção criteriosa para evitar redundância e ruído.",
        "condicao": "Integrar por `id_escola` ↔ `CO_ENTIDADE` e validar cobertura/chaves."
    },
    {
        "fonte": "INSE 2023",
        "grupo_de_features": "MEDIA_INSE, INSE_CLASSIFICACAO e percentuais por nível socioeconômico",
        "decisao": "Incluir futuramente",
        "justificativa": "Indicadores socioeconômicos em nível escolar são relevantes para explicar desigualdades educacionais e tendem a ser disponíveis antes da predição.",
        "condicao": "Integrar por `id_escola` ↔ `ID_ESCOLA` e validar nulos/cobertura."
    }
])

fontes_complementares


## 10. Pontos encaminhados para TC3-42

Os seguintes pontos devem ser tratados pela análise específica de Data Leakage:

- Verificar se `proficiencia` é insumo legítimo no momento de predição ou se é parte direta/indireta da definição do target `alfabetizado`.
- Avaliar se `presenca`, `preenchimento_caderno` e `caderno` carregam informação posterior ou ligada ao instrumento de avaliação.
- Confirmar se os indicadores municipais de ano anterior foram calculados estritamente com dados anteriores à observação do aluno.
- Avaliar se códigos de escola/município, quando usados diretamente, causam memorização indevida ou vazamento por segmentação extrema.

Até a conclusão do TC3-42, variáveis marcadas como **Investigar** não devem ser consideradas features definitivas.


## 11. Pontos encaminhados para pré-processamento posterior

Itens fora do escopo do TC3-41 e que devem ser tratados em tickets posteriores:

- Estratégia de imputação para indicadores municipais, `peso_aluno` e `proficiencia`.
- Encoding de variáveis categóricas (`sigla_uf`, `rede`, eventual uso de escola/município).
- Scaling de variáveis numéricas, se exigido pelo algoritmo escolhido.
- Criação de pipeline reprodutível de transformação.
- Integração física das bases Censo Escolar 2023 e INSE 2023.
- Treinamento, tuning e avaliação de modelos.


## 12. Parecer técnico

A seleção inicial de features deve priorizar variáveis disponíveis antes da predição, com interpretação educacional clara e baixo risco de vazamento.

**Candidatas principais da `ml_aluno`:** `ano_referencia`, `serie`, `sigla_uf`, `rede`, indicadores municipais de ano anterior e chaves territoriais/escolares usadas com cautela.

**Excluídas da modelagem como features:** identificadores individuais, descrições redundantes, target, split e timestamp técnico.

**Pendentes de análise de leakage:** `proficiencia`, `caderno`, `presenca`, `preenchimento_caderno` e uso direto de identificadores de escola/município.

**Fontes futuras recomendadas:** INSE 2023 e Censo Escolar 2023, desde que integradas com validação de chaves, cobertura e temporalidade.
